# अध्याय 7: चॅट अनुप्रयोग तयार करणे
## Microsoft Foundry Models API जलद प्रारंभ

हा नोटबुक [Azure OpenAI नमुने संग्रह](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) मधून रूपांतरित केला आहे ज्यामध्ये [Azure OpenAI](notebook-azure-openai.ipynb) सेवा प्रवेश करणार्‍या नोटबुकचा समावेश आहे.

> **नोट:** GitHub Models जुलै 2026 च्या अखेरीस सेवा थांबवत आहे. हा नोटबुक आता [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) कडे लक्ष देतो, जे समान मोफत चाचणीसाठी मोडेल कॅटलॉग आणि Azure AI इन्फरन्स SDK अनुभव प्रदान करतो.


# आढावा  
"मोठे भाषा मॉडेल हे टेक्स्टला टेक्स्टमध्ये नकाशित करणारे फंक्शन्स आहेत. टेक्स्टच्या इनपुट स्ट्रिंग दिल्यास, एक मोठे भाषा मॉडेल पुढे येणारा टेक्स्ट काय असेल याचा अंदाज लावण्याचा प्रयत्न करते"(1). हा "क्विकस्टार्ट" नोटबुक वापरकर्त्यांना उच्च-स्तरीय LLM संकल्पना, AML सह सुरू करण्यासाठी मुख्य पॅकेज गरजा, प्रॉम्प्ट डिझाइनसाठी सौम्य परिचय, आणि वेगवेगळ्या वापराच्या प्रकरणांचे अनेक लहान उदाहरणे याची ओळख करून देईल. 


## मजकूर सूची  

[आढावा](#overview)  
[OpenAI सेवा कशी वापरायची](#how-to-use-openai-service)  
[1. तुमची OpenAI सेवा तयार करणे](#1.-creating-your-openai-service)  
[2. स्थापनेची प्रक्रिया](#2.-installation)    
[3. क्रेडेन्शियल्स](#3.-credentials)  

[वापर प्रकरणे](#use-cases)    
[1. मजकूर सारांशित करा](#1.-summarize-text)  
[2. मजकूर वर्गीकरण करा](#2.-classify-text)  
[3. नवीन उत्पादन नावे तयार करा](#3.-generate-new-product-names)  
[4. वर्गीकरण करणारा फाइन-ट्यून करा](#4.fine-tune-a-classifier)  

[संदर्भ](#references)


### आपला पहिला प्रॉम्प्ट तयार करा  
हा लहान सराव Microsoft Foundry Models मध्ये एका साध्या कार्यासाठी "सारांश" मॉडेलमध्ये प्रॉम्प्ट सबमिट करण्यासाठी मूलभूत परिचय प्रदान करेल.


**टप्पे**:  
1. आपल्याकडे नसेल तर आपल्या पायथन वातावरणात `azure-ai-inference` लायब्ररी इंस्टॉल करा.  
2. मानक सहाय्यक लायब्ररी लोड करा आणि आपले Microsoft Foundry Models क्रेडेन्शियल सेट करा.  
3. आपल्या कार्यासाठी एक मॉडेल निवडा  
4. मॉडेलसाठी एक सोपा प्रॉम्प्ट तयार करा  
5. आपल्या विनंतीस मॉडेल API कडे सबमिट करा!


### 1. `azure-ai-inference` इन्स्टॉल करा


In [ ]:
%pip install azure-ai-inference

### 2. मदतनीस लायब्ररी आयात करा व प्रमाणीकरण तयार करा


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. योग्य मॉडेल शोधणे  
GPT-4o आणि GPT-4o मिनी सारखे मॉडेल नैसर्गिक भाषा समजू शकतात आणि तयार करू शकतात, आणि ते Microsoft Foundry Models कॅटलॉगमध्ये Meta, Mistral, Cohere, आणि Microsoft कडून आलेल्या मॉडेल्ससह उपलब्ध आहेत.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. प्रॉम्प्ट डिझाइन  

"मोठ्या भाषा मॉडेल्सचे जादू म्हणजे मोठ्या प्रमाणावर मजकूरावर या भाकीत त्रुटी कमी करण्यासाठी प्रशिक्षण घेऊन, मॉडेल्स या भाकीतांसाठी उपयुक्त संकल्पना शिकतात. उदाहरणार्थ, ते खालीलप्रमाणे संकल्पना शिकतात"(1):

* कसे शब्दांची हिज्जे करावी
* व्याकरण कसे कार्य करते
* कसे पुनःप्रस्तावना करावी
* प्रश्न कसे उत्तर द्यावेत
* संभाषण कसे चालवावे
* अनेक भाषांमध्ये कसे लिहावे
* कोड कसे करावे
* इत्यादी.

#### मोठ्या भाषा मॉडेलचे नियंत्रण कसे करावे  
"मोठ्या भाषा मॉडेलमध्ये सर्व इनपुटपैकी सर्वाधिक प्रभावी इनपुट म्हणजे मजकूर प्रॉम्प्ट(1).

मोठ्या भाषा मॉडेल्स अनेक पद्धतींनी प्रॉम्प्ट करून आउटपुट तयार करू शकतात:

सूचना: तुम्हाला काय हवे ते मॉडेलला सांगा
पूर्णता: तुम्हाला हवे असलेले सुरुवातीचे पूर्ण करण्यासाठी मॉडेलला प्रेरित करा
प्रदर्शन: मॉडेलला तुम्हाला काय हवे ते दाखवा, खालीलपैकी एखाद्या स्वरूपात:
प्रॉम्प्टमध्ये काही उदाहरणे
फाईन-ट्युनिक प्रशिक्षण डेटासेटमध्ये शेकडो किंवा हजारो उदाहरणे"



#### प्रॉम्प्ट तयार करण्यासाठी तीन मूलभूत मार्गदर्शक तत्त्वे आहेत:

**दाखवा आणि सांगा**. तुम्हाला काय हवे आहे ते सूचना, उदाहरणे, किंवा दोन्हींच्या संयोजनाद्वारे स्पष्ट करा. जर तुम्हाला मॉडेलने वस्तूंची यादी अक्षरक्रमाने क्रमवारी लावावी किंवा परिच्छेदाचे भावना वर्गीकृत करावे असे वाटत असेल, तर तेच दाखवा.

**चांगला डेटा द्या**. जर तुम्ही एक वर्गीकारण प्रणाली तयार करत असाल किंवा मॉडेलने एखाद्या स्वरूपाचे पालन करावे असे अपेक्षित असेल, तर पुरेसे उदाहरणे असावीत याची खात्री करा. तुमची उदाहरणे काळजीपूर्वक तपासा — मॉडेल सामान्यतः मूलभूत शब्दलेखन चुका ओळखून उत्तर देते, पण कधी कधी ते हे जाणून घेतो की ही कठोर इच्छा आहे आणि यामुळे त्याच्या प्रतिसादावर परिणाम होऊ शकतो.

**तुमची सेटिंग्स तपासा**. temperature आणि top_p सेटिंग्ज प्रतिसाद तयार करताना मॉडेल किती निश्चित आहे हे नियंत्रित करतात. जर तुम्ही असलेला प्रतिसाद असा विचारत असाल ज्याचा फक्त एक योग्य उत्तर आहे, तर तुम्हाला हे कमी करायचे आहेत. जर तुम्हाला अधिक विविध प्रतिसाद पाहिजेत, तर तुम्ही हे जास्त ठेऊ शकता. लोकांनी या सेटिंग्जबाबतची सर्वात मोठी चूक म्हणजे त्यांना "चातुर्य" किंवा "सर्जनशीलता" नियंत्रण समजणे.


स्रोत: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. सादर करा!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### त्याच कॉलची पुनरावृत्ती करा, परिणामांची तुलना कशी होते? 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## मजकूर संक्षेप करा  
#### आव्हान  
मजकूराच्या शेवटी 'tl;dr:' असे जोडून मजकूर संक्षेप करा. लक्षात घ्या की मॉडलशिवाय कोणत्याही अतिरिक्त सूचनांशिवाय अनेक कार्ये कशी पार पाडायची हे कसे समजते. तुम्ही मॉडेलची कार्ये बदलण्यासाठी आणि तुम्हाला मिळणाऱ्या संक्षेपाला सानुकूल करण्यासाठी tl;dr पेक्षा अधिक वर्णनात्मक कमांडसह प्रयोग करू शकता (3).  

अलीकडील कामात मोठ्या प्रमाणात लेखन संकलनावर पूर्व-प्रशिक्षण करून आणि नंतर विशिष्ट कार्यावर सूक्ष्मसुधारणा करून अनेक NLP कार्ये आणि मापन चाचण्यांमध्ये मोठे प्रगती दाखवण्यात आली आहे. जरी वास्तुकलामध्ये सहसा कार्य-निरपेक्ष असले तरी, या पद्धतीला हजारो किंवा दहाजणो उदाहरणांच्या कार्य-विशिष्ट सूक्ष्मसुधारणा डेटासेटची आवश्यकता असते. विरुद्धपणे, माणूस साध्या सूचनांवरून किंवा काही उदाहरणांवरून नवीन भाषा कार्य सहसा पार पाडू शकते - हे सध्याच्या NLP प्रणाली अजूनही मोठ्या प्रमाणात करत नाहीत. येथे आम्ही दाखवतो की भाषा मॉडेल्सचा विस्तार केल्याने कार्य-निरपेक्ष, काही-उदाहरणीनंत्री कार्यप्रदर्शन मोठ्या प्रमाणात सुधारते, कधीकधी तर पूर्वीच्या अत्याधुनिक सूक्ष्मसुधारणा पद्धतींसह स्पर्धात्मकही होते.  



संक्षेप  


# अनेक वापर प्रकरणांसाठी व्यायाम  
1. मजकूर सारांश करा  
2. मजकूर वर्गीकृत करा  
3. नवीन उत्पादनांची नावे तयार करा


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## मजकूर वर्गीकरण  
#### आव्हान  
इन्फरन्स वेळेस दिलेल्या श्रेणीत वस्तू वर्गीकरण करा. खालील उदाहरणात, आम्ही दोन्ही श्रेणी आणि वर्गीकरणासाठी मजकूर प्रॉम्प्टमध्ये (*playground_reference) पुरवतो. 

ग्राहक चौकशी: नमस्कार, माझ्या लॅपटॉपच्या कीबोर्डवरील एका की तुटली आहे आणि मला त्याची रिप्लेसमेंट हवी आहे:

वर्गीकृत श्रेणी:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## नवीन उत्पादन नावे तयार करा
#### आव्हान
उदाहरण शब्दांमधून उत्पादन नावे तयार करा. येथे आम्ही उत्पादनासाठी नाव तयार करताना ती माहिती प्रॉम्प्टमध्ये देतो. आम्ही नमुना देखील देतो ज्यामुळे आपल्याला हवा असलेला नमुना कसा दिसेल हे समजेल. याद्वारे आम्ही तापमान मूल्य जास्त ठेवले आहे ज्यामुळे अधिक अनियमितता आणि नाविन्यपूर्ण प्रतिसाद मिळतात.

उत्पादन वर्णन: घरगुती मिल्कशेक बनविण्याचे साधन
बीज शब्द: वेगवान, आरोग्यदायी, संक्षिप्त.
उत्पादन नावे: HomeShaker, Fit Shaker, QuickShake, Shake Maker

उत्पादन वर्णन: असे जोडीचे बूट जे कोणत्याही पायाच्या आकाराला बसू शकतील.
बीज शब्द: अनुकूलनीय, योग्य, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# संदर्भ  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [टेक्स्ट वर्गीकरणासाठी GPT मॉडेल्सचे सूक्ष्मसुधारणे करण्यासाठीच्या सर्वोत्तम पद्धती](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# आणखी मदतीसाठी  
[OpenAI व्यावसायिकरण टीम](AzureOpenAITeam@microsoft.com) 


# योगदानकर्ते
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
